In [ ]:
import jax.numpy as jnp



import matplotlib.pyplot as plt

import seaborn as sns
%load_ext autoreload
%autoreload 2
import scipy.stats as sts
from shapemetrics import paths

paths.set_figure("Figure4")
import numpy as np
import pandas as pd
from scipy.spatial.distance import squareform
from scipy.cluster import hierarchy
from sklearn.manifold import MDS
from sklearn.decomposition import PCA

import tqdm
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from pathlib import Path
import pickle

import matplotlib as mpl
new_rc_params = {'text.usetex': False,
"svg.fonttype": 'none'
}
mpl.rcParams.update(new_rc_params)

### FIGURES 1-4 MAIN IBL ANALYSIS

In [ ]:
def convert_to_rw_choices(cs_trials):
    choices = []
    for subject in cs_trials:
        left = subject[:,:5] == -1
        right = subject[:,5:] == 1
        both = np.column_stack([left.astype(int), right.astype(int)])
        print(both.shape)
        choices.append(np.mean(both,0))
    return np.array(choices)

def distances_mantel(dist1, dist2, n_perm=1000, seed=None):
    corrs_shuffle = []
    S = dist1.shape[0]
    flat_dist1 = dist1[np.triu_indices(S,1)]
    rng = np.random.default_rng(seed)
    for i in range(n_perm):
        perm = rng.permutation(S)
        shuffled_dist2 = dist2[perm, :][:, perm]
        flat_dist2_s = shuffled_dist2[np.triu_indices(S,1)]
        r_s,_ = sts.pearsonr(flat_dist2_s, flat_dist1)
        corrs_shuffle.append(r_s)
    return corrs_shuffle


In [ ]:
# Load the data
DATA_DIR = paths.derived("iblreproducibility", "data_fig")
all_dist_neural = np.load(DATA_DIR / "all_dist_neural.npy", allow_pickle=True)
all_dist_cc = np.load(DATA_DIR / "all_dist_cc.npy", allow_pickle=True)
all_c_corrs = np.load(DATA_DIR / "all_c_corrs.npy", allow_pickle=True)

In [ ]:
# Load the data - no cv
DATA_DIR = paths.derived("iblreproducibility", "data_fig")
dist_neural_nocv = np.load(DATA_DIR / "dist_neural_nocv.npy", allow_pickle=True)
dist_cc_nocv = np.load(DATA_DIR / "dist_cc_nocv.npy", allow_pickle=True)
choices = np.load(DATA_DIR / "choices.npy", allow_pickle=True)

In [ ]:
# FIGURE 1-4
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(7*1.5,7*1.5), constrained_layout=True)
sns.despine(left=True, bottom=True)

# Adding to the figure 
examples_idx= [10,12,19]
axes = axes.flatten()

### First - PSYCHOMETRIC CURVES
contrast = [-100.  ,  -25.  ,  -12.5 ,   -6.25,    0.  ,    6.25,   12.5 ,
         25.  ,  100.  ]

axes[1].plot(choices.T, color='gray', alpha=0.5)
axes[1].plot(np.nanmean(choices,0), lw=4)   
axes[1].set_ylabel("P(right)")
axes[1].set_xlabel("stimulus contrast")
axes[1].set_xticks(range(len(contrast)),contrast,rotation=45)
axes[1].set_yticks([0,0.5,1],[0,'.5',1])
axes[1].set_ylim(0,1)
axes[1].set_xlim(0,8)
sns.despine(bottom=False, left=False, ax=axes[1])

data_corrs = np.array(all_c_corrs)[:,0]
dist_neural = all_dist_neural[np.argmax(data_corrs)].copy()
dist_cc = all_dist_cc[np.argmax(data_corrs)].copy()

### Second - DISTANCE MATRICES
dist_neural = np.asarray(dist_neural, dtype=float)
dist_cc = np.asarray(dist_cc, dtype=float)
dist_neural_ = dist_neural.copy()
dist_cc_ = dist_cc.copy()
dist_neural_[np.triu_indices(dist_neural.shape[0], 0)] = np.nan
dist_cc_[np.tril_indices(dist_neural.shape[0], 0)] = np.nan

axes[2].imshow(dist_neural_,cmap="Grays")

axes[2].set_xlabel("neural distances", fontsize=11)

axes[2].set_title('behavioral distances', fontsize=11)
axes[2].imshow(dist_cc_,cmap="Purples")
axes[2].set_xticks([])
axes[2].set_yticks([])
axes[2].tick_params(top=True, labeltop=True, bottom=False, labelbottom=False,left=False,labelleft=False,right=True,labelright=True)

#change ticks fontsize
for ti,tick in enumerate(axes[2].get_xticklabels()):
    tick.set_fontsize(8)
    axes[2].get_yticklabels()[ti].set_fontsize(8)
    #axes[1].get_xticklabels()[ti].set_fontsize(8)
    #axes[1].get_yticklabels()[ti].set_fontsize(8)


### Third - SCATTER PLOT OF DISTANCES
# take upper triangle for the non-cross-validated data
S = dist_neural_nocv.shape[0]
flat_dist_neural = dist_neural_nocv[jnp.triu_indices(S,1)]
flat_dist_cc = dist_cc_nocv[jnp.triu_indices(S,1)]

r_dist, p_dist = sts.pearsonr(flat_dist_cc,flat_dist_neural)

axes[3].scatter( flat_dist_cc,flat_dist_neural,color="k",facecolors='gray')
axes[3].text(1.5,0.1,"r={:.2f}\np={:.2f}".format(r_dist,p_dist),color="red")
axes[3].set_xlabel("behavioral\ndistance", fontsize=11)
axes[3].set_ylabel("neural\ndistance", fontsize=11)
axes[3].set_xticks([])
axes[3].set_yticks([])
axes[3].set_title("distances correlation\n(non-cross-validated)", fontsize=11)
axes[3].scatter(dist_cc[examples_idx,:][:,examples_idx][np.triu_indices(len(examples_idx),1)].flatten(),
                dist_neural[examples_idx,:][:,examples_idx][np.triu_indices(len(examples_idx),1)].flatten(),facecolors='none', edgecolors='red',zorder=10)
sns.despine(bottom=False, left=False, ax=axes[3])

### Fourth - DISTRIBUTION OF Z SCORES
from scipy.stats import norm
r = np.array(all_c_corrs[:, 0])
null_means = np.array([np.mean(c) for c in all_c_corrs[:, 1]])
null_stds = np.array([np.std(c) for c in all_c_corrs[:, 1]])

# Calculate Z-scores
z = (r - null_means) / null_stds

print(f'Mean {z.mean()}, cdf {norm.cdf(z.mean())}')

sns.histplot(
    z, 
    bins=20, 
    kde=True, 
    color="steelblue", 
    edgecolor="black", 
    alpha=0.7,
    ax=axes[4]
) 

#sns.kdeplot(null_means, color='orange', linewidth=2, alpha=0.7, ax=ax)

axes[4].axvline(0, color='gray', linestyle='--', linewidth=2, label='Null Expectation')
axes[4].axvline(np.mean(z), color='firebrick', linestyle='-', linewidth=2, label=f'Mean Z = {np.mean(z):.2f}')

axes[4].set_yticks([])
axes[4].set_yticklabels([])
axes[4].set_ylabel('')
axes[4].set_xlabel('z-score', fontsize=11)
axes[4].set_title('z-score distribution across\n 100 cv folds', fontsize=12)
axes[4].legend(frameon=True, facecolor='white')
sns.despine(bottom=False, left=False, ax=axes[4])

plt.tight_layout()
plt.savefig('figure_ibl/IBL_distances.svg',dpi=300)
plt.show()

### Distances across regions

In [ ]:
# Load the data
DATA_DIR = paths.derived("iblreproducibility", "data_fig")
means_dist_subject = np.load(DATA_DIR / "means_dist_subject.npy", allow_pickle=True)

In [ ]:
import matplotlib.colors as mcolors
means_dist = np.nanmean(means_dist_subject, axis=0)
regions = np.array(['CA1', 'DG', 'LP', 'PO', 'VISa'])
p_cell = np.full((len(regions), len(regions)), np.nan)

for r, reg in enumerate(regions):
    for c in range(r+1, len(regions)):
        mean_cc = means_dist[c, c]
        mean_rr = means_dist[r, r]

        dist = means_dist_subject[:, r, c]
        # test whether the distribution of the d(r,c) across subjects is significantly bigger than the distribution of the biggest between r and c
        if mean_cc > mean_rr:
            t, p = sts.ttest_rel(dist, means_dist_subject[:, c, c], alternative='greater')
            # or should i do wilcoxon?
            w, p_w = sts.wilcoxon(dist, means_dist_subject[:, c, c], alternative='greater')
        else:
            t, p = sts.ttest_rel(dist, means_dist_subject[:, r, r], alternative='greater')
            w, p_w = sts.wilcoxon(dist, means_dist_subject[:, r, r], alternative='greater')
        p_cell[r, c] = p_w

means_dist[np.tril_indices(len(regions))] = np.nan

sig_matrix = np.where(p_cell < 0.05, 1.0, 0.0)

# Re-apply NaNs to the lower triangle and diagonal
sig_matrix[np.isnan(p_cell)] = np.nan

# Define two flat colors: [Color for False, Color for True]
cmap = mcolors.ListedColormap(['lightgrey', 'mediumseagreen'])

plt.figure(figsize=(6, 4))
sns.heatmap(sig_matrix, annot=means_dist, fmt=".2f", cmap=cmap, vmin=0, vmax=1,
            yticklabels=regions, xticklabels=regions, cbar=False)

plt.show()

### FIGURE 5 is the per-region correlation distibution plot

In [ ]:
# Load
DATA_DIR = paths.derived("iblreproducibility", "data_fig")
region_info = pd.read_csv(DATA_DIR / 'region_info.csv')

In [ ]:
region_info.groupby('region').agg({'corr': ['mean', 'std'], 'z-score': ['mean', 'std']})

In [ ]:
region_info
sns.histplot(
    data=region_info, 
    x="z-score", 
    hue="region", 
    bins=10, 
    kde=True, 
    palette=sns.color_palette("Set2", len(regions)),
    edgecolor="black", 
    alpha=0.7
)

In [ ]:
# FIGURE 5 
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(7*1.5,7*1.5), constrained_layout=True)

axes = axes.flatten()

colors = [ '#03A6A6', '#88D94E', '#88D94E', '#F28D9F', '#F28D9F']
palette = {'VISa': '#03A6A6', 'CA1': '#88D94E', 'DG': '#88D94E', 'LP': '#F28D9F', 'PO': '#F28D9F'}
#
# colors = ['#03A6A6', '#03A6A6', '#03A6A6', '#FF90FF', '#F28D9F']
# Plot distribution of correlations for each region
axes[5].set_title("z-scores across folds", fontsize=11, pad=10)

# Use palette only (remove explicit 'color' to let palette apply correctly)
sns.violinplot(x="region", y="z-score", data=region_info, order=regions, palette=palette, ax=axes[5], inner='quartile', linewidth=1)
axes[5].axhline(0, color='k', linewidth=.8)
axes[5].set_ylabel("z-score", fontsize=12)
#axes[5].set_yticks([0, 0.35])
#axes[5].set_ylim(-0.1, 0.38)

# Add significance stars where the lower 2.5% percentile of the distribution > 0
"""for i, region in enumerate(regions):
    vals = region_info.loc[region_info['region'] == region, 'z-score'].values
    if len(vals) == 0:
        continue
    lower = np.nanpercentile(vals, 2.5)

    if lower > 0:
        # compute a y position just above the data for this region
        ylim = axes[5].get_ylim()
        data_max = np.nanmax(vals)
        margin = 0.02 * (ylim[1] - ylim[0])
        y = data_max + margin
        # expand ylim if needed to make the star visible
        if y > ylim[1] - margin:
            axes[5].set_ylim(ylim[0], y + margin)
        axes[5].text(i, y+0.05, '*', ha='center', va='bottom', fontsize=14, color='k')
"""
plt.tight_layout()
sns.despine(bottom=True)

### Now, the last 3 figures are from ASD analysis

In [ ]:
import pandas as pd
from shapemetrics import paths

paths.set_figure("Figure4")
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
from scipy.cluster import hierarchy
from collections import defaultdict
from matplotlib.lines import Line2D
from scipy.stats import ttest_rel

%load_ext autoreload
%autoreload 2
from matplotlib.colors import Normalize

In [ ]:
# Load 
DATA_DIR = paths.derived("iblreproducibility", "data_fig")
dist_neural_asd = np.load(DATA_DIR / "dist_neural_asd.npy", allow_pickle=True)
labels_arr = np.load(DATA_DIR / "labels_arr.npy", allow_pickle=True)

In [ ]:
# FOR THE STATISTICAL TEST OF DIFFERENCES: get the neural distances and compute:
# 1. distance matrix within control (mean for each control subject to each other control subject)
# 2. distance matrix between each control and each other genotype (so, each control and the average distance between subjects of a genotype)
# 3. now you have #ControlSubjects references and 4 variations 

def within_distance(dist_matrix, labels, genotype):
    mask_g = labels == genotype
    within = dist_matrix[np.ix_(mask_g, mask_g)]
    np.fill_diagonal(within, np.nan) 
    return within

def across_distance(dist_matrix, labels, g1, g2):
    mask_g1 = labels == g1
    mask_g2 = labels == g2
    return dist_matrix[np.ix_(mask_g1, mask_g2)]

# Get mean distances for each control (first dimension) subject
control = np.nanmean(within_distance(dist_neural_asd, labels_arr, 'N'), axis=1)
control_vs_F = across_distance(dist_neural_asd, labels_arr, 'N', 'F').mean(axis=1)
control_vs_C = across_distance(dist_neural_asd, labels_arr, 'N', 'C').mean(axis=1)
control_vs_S = across_distance(dist_neural_asd, labels_arr, 'N', 'S').mean(axis=1)

# create a dataframe for plotting
df_control_distances = pd.DataFrame({
    'control_subject': np.arange(len(control)),
    'Within control': control,
    'Control vs S': control_vs_S,
    'Control vs F': control_vs_F,
    'Control vs C': control_vs_C,
})

In [ ]:
def plot_mds_pca(dist_matrix, labels, n_components=50, ax=None):

    # DIstance matrices are not euclidean so PCA directly is not ideal
    # MDS on the distance matrix
    mds = MDS(n_components=n_components, 
              dissimilarity='precomputed', 
              random_state=66,
              normalized_stress=False,
              n_init=10,)
    
    mds_coords = mds.fit_transform(dist_matrix)

    #mds_coords = (mds_coords - mds_coords.mean(axis=0)) / mds_coords.std(axis=0)

    # PCA on MDS coordinates
    pca_2d = PCA(n_components=2, random_state=42)
    pca_coords = pca_2d.fit_transform(mds_coords)

    colors = {'N':'#0D0D0D', 'F':'#61A656', 'C':'#D99C2B', 'S':'#A67232'}
    map_labels = {'C':'Cntnap2', 'F':'Fmr1', 'N':'C57BL6', 'S':'Shank3'}

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    # PCA on MDS plot
    for g in np.unique(labels):
        mask = np.array(labels) == g
        ax.scatter(pca_coords[mask, 0], pca_coords[mask, 1],
                        c=colors[g], label=map_labels[g], s=60, edgecolors='white', linewidths=0.5)

    ax.set_xticks([])
    ax.set_xticklabels([])

    ax.set_yticks([])
    ax.set_yticklabels([])

    ax.legend()

    sns.despine(ax=ax)

    return ax, pca_coords

In [ ]:
def genotype_perm_test(D, labels, g1, g2, n_perm=10000, seed=0):
    """Is the g1-g2 distance larger than the within-g1 distance?
    Null: shuffle the genotype label attached to each subject."""
    rng = np.random.default_rng(seed)
    def stat(lab):
        w = D[np.ix_(lab == g1, lab == g1)].astype(float)
        np.fill_diagonal(w, np.nan)
        return np.nanmean(D[np.ix_(lab == g1, lab == g2)]) - np.nanmean(w)
    obs = stat(labels)
    null = np.array([stat(rng.permutation(labels)) for _ in range(n_perm)])
    return obs, (np.sum(null >= obs) + 1) / (n_perm + 1)

def control_perm_test(D, labels, g1, g2, n_perm=10000, seed=0):
    """Is the control-g2 distance larger than the control-g1 distance?
    Just like the test above, but instead of comparing the mean of clusters, 
    compare the mean distance of each control and the mean for other genotype
    for different permutations
    Null: shuffle the genotype label attached to each subject."""
    rng = np.random.default_rng(seed)
    def stat(lab):
        w = within_distance(D, lab, g1)
        across = across_distance(D, lab, g1, g2)
        diff_control = np.nanmean(across, axis=1) - np.nanmean(w, axis=1) # mean for each control subjects
        return diff_control.mean()  # mean across all control subjects
    obs = stat(labels) 
    null = np.array([stat(rng.permutation(labels)) for _ in range(n_perm)])
    return obs, (np.sum(null >= obs) + 1) / (n_perm + 1)

In [ ]:
## Figure 6-8

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(7*1.5,7*1.5), constrained_layout=True)
axes = axes.flatten()

# FIG 6 - HIERARCHICAL CLUSTERING AND ORDERING (drawn inside axes[6])
colors = {'N':'#0D0D0D', 'F':'#61A656', 'C':'#D99C2B', 'S':'#A67232'}
genotypes = ['N', 'F', 'C', 'S']
contrast_names =    ['cL100', 'cL025', 'cL012', 'cL006', 'c0', 'cR006', 'cR012', 'cR025', 'cR100']

# FIG 7 - PCA (placed in axes[7])
ax_pca, _ = plot_mds_pca(dist_neural_asd, labels_arr, ax=axes[7])

# FIG 6 - ORDERED DISTANCE MATRIX + DENDROGRAM (place inside axes[6])
from scipy.cluster import hierarchy
import matplotlib.colors as mcolors
from matplotlib import gridspec

S = dist_neural_asd.shape[0]
condensed = dist_neural_asd[np.triu_indices(S,1)]
linkage = hierarchy.ward(condensed)
linkage = hierarchy.optimal_leaf_ordering(linkage, condensed)
leaf_order = hierarchy.leaves_list(linkage)
ordered_dist = dist_neural_asd[leaf_order, :][:, leaf_order]

# Replace axes[6] with two stacked subaxes (dendrogram above heatmap)
axes[6].clear()
axes[6].set_xticks([])
axes[6].set_yticks([])
sns.despine(ax=axes[6], left=True, bottom=True)
gs_inner = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=axes[6].get_subplotspec(), height_ratios=[1, 4], hspace=0.02)
dendro_ax = fig.add_subplot(gs_inner[0])
heatmap_ax = fig.add_subplot(gs_inner[1])

# Dendrogram (top)
dn = hierarchy.dendrogram(linkage, ax=dendro_ax, no_labels=True, color_threshold=0, above_threshold_color='k')
dendro_ax.axis('off')

# Heatmap (bottom)
heatmap_ax.imshow(ordered_dist, cmap='gray', aspect='auto', vmin=np.nanmin(ordered_dist), vmax=np.nanmax(ordered_dist), origin='lower')
heatmap_ax.set_xticks([])
heatmap_ax.set_yticks([])
for spine in heatmap_ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

# Small genotype color strip aligned with rows of the heatmap
ordered_labels = labels_arr[leaf_order]
lab_to_color = colors
color_list = mcolors.to_rgba_array([lab_to_color[lab] for lab in ordered_labels])
pos2 = heatmap_ax.get_position()
strip_width = 0.02
pad = 0.005
new_heatmap_left = pos2.x0 + strip_width + pad
new_heatmap_width = max(pos2.width - (strip_width + pad), pos2.width * 0.5)
heatmap_ax.set_position([new_heatmap_left, pos2.y0, new_heatmap_width, pos2.height])
cax = fig.add_axes([pos2.x0, pos2.y0, strip_width, pos2.height])
cax.imshow(color_list.reshape(-1,1,4), aspect='auto', origin='lower')
cax.set_xticks([])
cax.set_yticks([])

## FIG 8 - PARALLEL T-TEST PLOT (axes[8])
cols = ['Within control', 'Control vs S', 'Control vs F', 'Control vs C']
x_labels = ['C57BL6', 'Shank3', 'Fmr1', 'Cntnap2']

for _, row in df_control_distances[cols].iterrows():
    axes[8].plot(range(len(cols)), row.values, color='#888888', alpha=0.8, linewidth=0.8, zorder=3)

means = df_control_distances[cols].mean(axis=0)
sem = df_control_distances[cols].sem(axis=0)

axes[8].fill_between(range(len(cols)), means - sem, means + sem,
                color='#222222', alpha=0.15, zorder=2)

mean_colors = ['#0D0D0D', '#61A656', '#D99C2B', '#A67232']
axes[8].plot(range(len(cols)), means.values, color='#222222', linewidth=2.5, zorder=3)

for xi, col in enumerate(cols):
    axes[8].scatter(xi, means[col], color=mean_colors[xi], s=80, zorder=4, edgecolor='white', linewidth=1.2)

results = {}
a = df_control_distances['Within control'].values
y_max_val = df_control_distances[cols].max().max()

for xi, col in enumerate(['Control vs S', 'Control vs F', 'Control vs C'], start=1):
    b = df_control_distances[col].values
    mask = (~np.isnan(a)) & (~np.isnan(b))
    if mask.sum() >= 2:
        t_stat, p_val = ttest_rel(a[mask], b[mask])
        print(t_stat, p_val)
        obs, p_val = genotype_perm_test(dist_neural_asd, labels_arr, 'N', col.split()[-1])
        print(p_val)

        results[col] = {'n': int(mask.sum()), 't': float(t_stat), 'p': float(p_val)}

        axes[8].text(xi, y_max_val - (0.05 * y_max_val), f'p = {p_val:.4g}', ha='center', va='bottom', fontsize=9, color=mean_colors[xi])

axes[8].set_xticks(range(len(cols)))
axes[8].set_xticklabels(x_labels, fontsize=10)
axes[8].set_yticks([])

axes[8].set_title('Distance from C57BL6 to', fontsize=11)

sns.despine(ax=axes[8], trim=True)
axes[8].grid(axis='y', color="#ECE9E9", linewidth=0.8, linestyle='--')
axes[8].grid(axis='x', visible=False)
axes[8].legend(frameon=False, loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

### Time resolved correlation

In [ ]:

import jax.numpy as jnp

import matplotlib as mpl
import matplotlib.pyplot as plt

new_rc_params = {'text.usetex': False,
"svg.fonttype": 'none'
}
mpl.rcParams.update(new_rc_params)

In [ ]:
### CLUSTER THE NEURAL DISTANCES, THEN LOOK AT BEHAVIOUR PER CLUSTER
# Same clustering as the genotype panel: ward linkage on the condensed upper
# triangle, with optimal leaf ordering. Cut at k clusters with fcluster.
from sklearn.metrics import silhouette_score

# defined here too so this cell runs on its own, without the psychometric cells
CONTRASTS = np.array([-100., -25., -12.5, -6.25, 0., 6.25, 12.5, 25., 100.])

# the non-cross-validated neural distances (same matrix panel d's scatter uses)
D_all = np.asarray(dist_neural_nocv, dtype=float)
D_all = (D_all + D_all.T) / 2.0            # enforce exact symmetry for linkage
S_all = D_all.shape[0]
condensed_n = D_all[np.triu_indices(S_all, 1)]
link_n = hierarchy.ward(condensed_n)
link_n = hierarchy.optimal_leaf_ordering(link_n, condensed_n)

# ---- how many clusters? ---------------------------------------------------
KS = range(2, 9)
sil = {}
for k in KS:
    lab = hierarchy.fcluster(link_n, k, criterion='maxclust')
    sil[k] = silhouette_score(D_all, lab, metric='precomputed')
k_opt = max(sil, key=sil.get)

print(f"ward clustering of the {S_all}-mouse neural distance matrix "
      f"(non-cross-validated)")
print(f"  {'k':>3}{'silhouette':>13}{'cluster sizes':>26}")
for k in KS:
    lab = hierarchy.fcluster(link_n, k, criterion='maxclust')
    sizes = ','.join(str(int((lab == c).sum())) for c in np.unique(lab))
    print(f"  {k:>3}{sil[k]:13.3f}{sizes:>26}{'   <- best' if k == k_opt else ''}")
print(f"\n  silhouette is computed on the precomputed distances; it peaks at "
      f"k = {k_opt}.")
print("  Note it is a descriptive index -- it always returns a best k, even for")
print("  data with no cluster structure.")

# ---- psychometric curves per cluster --------------------------------------
K_SHOW = [2, 3, 4]
PALETTE = ['#1D3557', '#C1121F', '#2A9D8F', '#E9A03B', '#6A4C93', '#7F7F7F']
MIN_N_MEAN = 3      # below this, plot the mice individually

fig, axes = plt.subplots(1, len(K_SHOW) + 1,
                         figsize=(3.4 * (len(K_SHOW) + 1), 3.6))

ax = axes[0]
ax.plot(list(KS), [sil[k] for k in KS], 'o-', color='0.3', lw=1.5, ms=5)
ax.plot(k_opt, sil[k_opt], 'o', color='#C1121F', ms=9, zorder=5)
ax.set_xlabel("number of clusters k")
ax.set_ylabel("silhouette (precomputed)")
ax.set_title(f"cluster count\nbest k = {k_opt}", fontsize=10)
ax.set_xticks(list(KS))
ax.set_box_aspect(1)
sns.despine(ax=ax)

for a, k in zip(axes[1:], K_SHOW):
    lab = hierarchy.fcluster(link_n, k, criterion='maxclust')
    for ci, c in enumerate(np.unique(lab)):
        m = lab == c
        Y = np.asarray(choices, dtype=float)[m]
        col = PALETTE[ci % len(PALETTE)]
        x = np.arange(len(CONTRASTS))
        if m.sum() < MIN_N_MEAN:
            # too few animals for a mean +/- SEM to mean anything -- show each
            # mouse, in the cluster's colour
            for r, row in enumerate(Y):
                a.plot(x, row, marker='o', ms=3, lw=1.2, color=col, alpha=.9,
                       label=(f"cluster {c} (n={m.sum()}, shown individually)"
                              if r == 0 else None))
        else:
            mu = np.nanmean(Y, axis=0)
            n = np.sum(np.isfinite(Y), axis=0)
            sem = np.nanstd(Y, axis=0, ddof=1) / np.sqrt(np.maximum(n, 1))
            a.errorbar(x, mu, yerr=sem, marker='o', ms=3.5, lw=1.5, capsize=2,
                       elinewidth=1, color=col,
                       label=f"cluster {c} (n={m.sum()})")
    a.set_xticks(range(len(CONTRASTS)))
    a.set_xticklabels([f"{v:g}" for v in CONTRASTS], rotation=45, fontsize=6.5)
    a.set_ylim(0, 1)
    a.set_yticks([0, .5, 1])
    a.set_xlabel("signed contrast (%)")
    a.set_ylabel("P(right)" if k == K_SHOW[0] else "")
    a.set_title(f"k = {k}" + ("   (best)" if k == k_opt else ""), fontsize=10)
    a.legend(frameon=False, fontsize=7)
    a.set_box_aspect(1)
    sns.despine(ax=a)

plt.tight_layout()
plt.savefig('figure_ibl/IBL_clusters_behaviour.svg', dpi=300, bbox_inches='tight')
plt.savefig('figure_ibl/IBL_clusters_behaviour.pdf', bbox_inches='tight')
plt.show()

# cluster membership for reference
for k in K_SHOW:
    lab = hierarchy.fcluster(link_n, k, criterion='maxclust')
    print(f"k={k}: " + ' '.join(f"{int(c)}:{int((lab==c).sum())}"
                                for c in np.unique(lab)))

In [ ]:
### DECODE GENOTYPE FROM THE NEURAL DISTANCE MATRIX (leave-one-out)
# One decoder: assign the held-out animal to whichever group its training
# members are closest to on average. Four one-vs-rest contrasts, each scored by
# balanced accuracy against a shuffled-label null of the same contrast.
from shapemetrics import paths

paths.set_figure("Figure4")
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path



DATA = paths.derived("iblreproducibility", "data_fig")
D = np.load(DATA / "dist_neural_asd.npy", allow_pickle=True).astype(float)
#D = (D + D.T) / 2
np.fill_diagonal(D, 0)
geno = np.asarray(np.load(DATA / "labels_arr.npy", allow_pickle=True)).astype(str)

NAME = {'N': 'C57BL6', 'F': 'Fmr1', 'C': 'Cntnap2', 'S': 'Shank3'}
COLOR = {'N': '#0D0D0D', 'F': '#61A656', 'C': '#D99C2B', 'S': '#A67232'}
N_PERM = 10000
RNG = np.random.default_rng(0)


def decode(Dsub, y):
    """Leave-one-out nearest-group. D[i,i] = 0, so the self term drops out of
    each sum for free and only the group counts need to exclude it."""
    g1, g0 = y.astype(float), (~y).astype(float)
    mean1 = (Dsub @ g1) / (g1.sum() - g1)      # mean distance to group 1
    mean0 = (Dsub @ g0) / (g0.sum() - g0)      # ... and to group 0
    return mean1 < mean0


def balanced_accuracy(y, pred):
    return 0.5 * (pred[y].mean() + (~pred[~y]).mean())


def balanced_accuracy_sem(y, pred):
    """SEM of balanced accuracy from the spread of hits across folds.

    Balanced accuracy is the mean of two group means, so its standard error is
    the two groups' standard errors added in quadrature and halved. Folds share
    most of their training data, so this understates the true uncertainty."""
    hit1 = pred[y].astype(float)              # 1 where a group-1 animal was called 1
    hit0 = (~pred[~y]).astype(float)          # 1 where a group-0 animal was called 0
    return 0.5 * np.hypot(hit1.std(ddof=1) / np.sqrt(len(hit1)),
                          hit0.std(ddof=1) / np.sqrt(len(hit0)))


def run(rows, y, name, key):
    Dsub = D[np.ix_(rows, rows)]
    pred = decode(Dsub, y)
    obs = balanced_accuracy(y, pred)
    sem = balanced_accuracy_sem(y, pred)
    null = np.empty(N_PERM)
    for t in range(N_PERM):
        ys = y[RNG.permutation(len(y))]
        null[t] = balanced_accuracy(ys, decode(Dsub, ys))
    p = (np.sum(null >= obs) + 1) / (N_PERM + 1)
    within = Dsub[np.ix_(y, y)][np.triu_indices(y.sum(), 1)].mean()
    across = Dsub[np.ix_(y, ~y)].mean()
    return dict(name=name, key=key, n1=int(y.sum()), n0=int((~y).sum()), acc=obs,
                sem=sem, null=null, p=p, within=within, across=across,
                ci=np.percentile(null, [5, 95]),
                sens=pred[y].mean(), spec=(~pred[~y]).mean())


# one-vs-rest for each genotype in turn, always against all 37 animals
rows = np.arange(len(geno))
res = [run(rows, geno == g, f"{NAME[g]} vs all others", g) for g in 'NFCS']

print(f"leave-one-out nearest-group decoding, {N_PERM} label shuffles per contrast\n")
print(f"  {'contrast':<24}{'n':>8}{'bal. acc':>10}{'SEM':>7}{'null':>8}"
      f"{'null 90% CI':>16}{'p':>8}{'sens':>7}{'spec':>6}{'within':>9}{'across':>8}")
for r in res:
    counts = f"{r['n1']} v {r['n0']}"
    ci = f"[{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]"
    print(f"  {r['name']:<24}{counts:>8}{r['acc']:>10.3f}{r['sem']:>7.3f}"
          f"{r['null'].mean():>8.3f}{ci:>16}{r['p']:>8.4f}{r['sens']:>7.2f}"
          f"{r['spec']:>6.2f}{r['within']:>9.1f}{r['across']:>8.1f}")
print("\n  The null interval is the 5th-95th percentile, so its upper edge is the")
print("  one-sided 5% level the p-values are computed against: a point above that")
print("  cap has p < 0.05.")
print("\n  'within' is the mean distance among the first group, 'across' its mean")
print("  distance to the second. The decoder needs the FIRST group to be tighter")
print("  with itself AND the second group likewise -- sens and spec split those")
print("  two halves, and a group that is internally diffuse drags spec down even")
print("  when the other group looks tight.")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))

ax = axes[0]
bars = ax.bar([j for j in range(len(res))], [r['acc'] for r in res],
              width=.62, color=[COLOR[r['key']] for r in res],
              edgecolor='white', linewidth=.6)
ax.errorbar(range(len(res)), [r['acc'] for r in res], yerr=[r['sem'] for r in res],
            fmt='none', ecolor='#C1121F', elinewidth=1.6, capsize=4,
            capthick=1.6, zorder=5)
for j, r in enumerate(res):
    ax.annotate(f"p = {r['p']:.4f}", (j, r['acc'] + r['sem']), ha='center',
                textcoords='offset points', xytext=(0, 6), fontsize=7,
                bbox=dict(boxstyle='square,pad=0.12', fc='white', ec='none'))
ax.axhline(.5, color='k', ls='--', lw=1, label="chance")
ax.set_xticks(range(len(res)))
ax.set_xticklabels([r['name'].replace(' vs ', '\nvs ') for r in res], fontsize=7)
ax.set_ylim(.3, max(r['acc'] + r['sem'] for r in res) + .12)
ax.set_ylabel("balanced accuracy $\\pm$ SEM")
ax.set_title(f"leave-one-out decoding\n(p vs {N_PERM} label shuffles)", fontsize=9)
ax.legend(frameon=False, fontsize=7)
sns.despine(ax=ax)

ax = axes[1]
w = .36
x = np.arange(len(res))
ax.bar(x - w / 2, [r['within'] for r in res], w, color='#C1121F',
       label="within group")
ax.bar(x + w / 2, [r['across'] for r in res], w, color='0.6', label="to the rest")
ax.set_xticks(x)
ax.set_xticklabels([r['name'].split(' vs ')[0] for r in res], fontsize=7)
ax.set_ylabel("mean neural distance")
ax.set_title("what the decoder is reading", fontsize=9)
ax.legend(frameon=False, fontsize=7)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('figure_ibl/ASD_genotype_decoding.svg', dpi=300, bbox_inches='tight')
plt.savefig('figure_ibl/ASD_genotype_decoding.pdf', bbox_inches='tight')
plt.show()

In [ ]:
### DO THE NEURAL CLUSTERS SEPARATE IN A JOINT PSYCHOMETRIC x RT SPACE?
# Panel f tests the choice curve and its inset tests RT, each on its own. This
# cell puts both on one plot: each mouse becomes a point, one axis summarising
# choice and one summarising speed, coloured by its neural cluster.
#
# Two questions, kept apart on purpose:
#   1. do the neural clusters separate better jointly than on either axis alone?
#      -> Mahalanobis distance between cluster centroids, against the same
#         cluster-label shuffle used in panel f
#   2. is there cluster structure in the behaviour at all, whatever the neural
#      clustering says? -> cluster the behaviour on its own and ask how well it
#         recovers the neural labels (adjusted Rand index, against a shuffle)
#
# Summaries are fixed in advance rather than chosen for separation: two for
# choice (where the animal sits, how steep it is), two for speed (how fast
# overall, how much it slows on hard trials). RT enters as a log, being
# right-skewed and positive.
from shapemetrics import paths

paths.set_figure("Figure4")
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster import hierarchy
from sklearn.metrics import adjusted_rand_score, silhouette_score

N_SHUF_J = 10000
_rj = np.random.default_rng(0)

# same clustering as panel f: ward on the symmetrised non-cross-validated
# neural distances, optimal leaf ordering, cut at k = 3
_Dj = np.asarray(dist_neural_nocv, float)
_Dj = (_Dj + _Dj.T) / 2.0
_cj = _Dj[np.triu_indices(len(_Dj), 1)]
_lab_j = hierarchy.fcluster(
    hierarchy.optimal_leaf_ordering(hierarchy.ward(_cj), _cj), 3,
    criterion='maxclust')
_keep_j = [c for c in np.unique(_lab_j) if (_lab_j == c).sum() >= 3]
_m_j = np.isin(_lab_j, _keep_j)
_gj = (_lab_j[_m_j] == _keep_j[1])

_rtz_j = np.load(DATA_DIR / "reaction_times.npz", allow_pickle=True)
_RTj = np.vstack([np.nanmedian(_rtz_j[f"arr_{i}"], axis=0)
                  for i in range(len(_rtz_j.files))])[_m_j]
_chj = np.asarray(choices, float)[_m_j]

# Accuracy is split by side as well as averaged. The two sides move in
# OPPOSITE directions when a curve shifts sideways -- a rightward shift helps
# right-side trials and hurts left-side ones -- so the average of the two
# cancels part of the effect and is the weaker summary, not the stronger one.
# 0% contrast is dropped from accuracy: reward is random there, so "correct"
# has no meaning.
_C = np.array([-100., -25., -12.5, -6.25, 0., 6.25, 12.5, 25., 100.])
_nz = [j for j, v in enumerate(_C) if v != 0]
_ACC = np.where(_C < 0, 1 - _chj, _chj)[:, _nz]
FEATS = {
    'rightward bias': _chj.mean(1) - .5,
    'mean accuracy': _ACC.mean(1),
    'accuracy, left': _ACC[:, :4].mean(1),
    'accuracy, right': _ACC[:, 4:].mean(1),
    'sensitivity': _chj[:, -1] - _chj[:, 0],
    'log median RT': np.log(np.median(_RTj, 1)),
    'RT modulation': np.log(_RTj[:, 4] / _RTj[:, [0, -1]].mean(1)),
}
PAIRS = [('rightward bias', 'log median RT'),
         ('mean accuracy', 'log median RT')]   # the scatters drawn below
XY = PAIRS[0]


def _shuffle_p(stat, X):
    obs = stat(X, _gj)
    null = np.array([stat(X, _gj[_rj.permutation(len(_gj))])
                     for _ in range(N_SHUF_J)])
    return obs, null, (np.sum(null >= obs) + 1) / (N_SHUF_J + 1)


def _absdiff(x, gv):
    return abs(x[gv].mean() - x[~gv].mean())


def _maha(X, gv):
    """Squared Mahalanobis distance between the two centroids, pooled within-
    cluster covariance. The multivariate analogue of the univariate contrast."""
    a, b = X[~gv], X[gv]
    d = b.mean(0) - a.mean(0)
    S = ((len(a) - 1) * np.cov(a.T) + (len(b) - 1) * np.cov(b.T)) / (len(a) + len(b) - 2)
    return float(d @ np.linalg.pinv(np.atleast_2d(S)) @ d)


print(f"n = {(~_gj).sum()} + {_gj.sum()} mice, {N_SHUF_J} cluster-label shuffles\n")
print(f"  {'summary':<16}{'cluster ' + str(_keep_j[0]):>12}"
      f"{'cluster ' + str(_keep_j[1]):>12}{'Cohen d':>10}{'p':>8}")
_uni = {}
for k, v in FEATS.items():
    _, _, p = _shuffle_p(_absdiff, v)
    d = (v[_gj].mean() - v[~_gj].mean()) / np.sqrt(
        (v[_gj].var(ddof=1) + v[~_gj].var(ddof=1)) / 2)
    _uni[k] = p
    print(f"  {k:<16}{v[~_gj].mean():>12.3f}{v[_gj].mean():>12.3f}{d:>10.2f}{p:>8.4f}")

print(f"\n  {len(FEATS)} summaries tested, uncorrected. The smallest p among "
      f"{len(FEATS)} is not")
print("  the same thing as a planned test: pick the measure before looking, or")
print("  read this table as descriptive.")
print("\n  joint tests (Mahalanobis between centroids)")
_joint = {}
for nm, keys in ((' + '.join(PAIRS[0]), list(PAIRS[0])),
                 ('bias + mean accuracy', ['rightward bias', 'mean accuracy']),
                 ('mean accuracy + log median RT',
                  ['mean accuracy', 'log median RT']),
                 ('accuracy right + log median RT',
                  ['accuracy, right', 'log median RT']),
                 ('all summaries', list(FEATS))):
    X = np.column_stack([FEATS[k] for k in keys])
    obs, null, p = _shuffle_p(_maha, X)
    _joint[nm] = p
    print(f"    {nm:<28} D2 = {obs:.3f}  null {null.mean():.3f}   p = {p:.4f}")

_r_xy = np.corrcoef(FEATS[XY[0]], FEATS[XY[1]])[0, 1]
print(f"\n  corr({XY[0]}, {XY[1]}) = {_r_xy:+.3f}")
print("  A second axis adds power only if it carries signal of its own, or is")
print("  correlated with the first in a way that sharpens the contrast. Neither")
print("  holds here, so the joint test pays for the extra dimension and gets")
print("  nothing back -- which is why its p is above the best univariate one.")

# ---- is there cluster structure in the behaviour at all? --------------------
Z = np.column_stack([FEATS[k] for k in FEATS])
Z = (Z - Z.mean(0)) / Z.std(0)
_cz = hierarchy.ward(Z)
print(f"\n  clustering the behaviour on its own ({Z.shape[1]} standardised summaries)")
print(f"    {'k':>3}{'silhouette':>13}{'ARI vs neural':>16}{'p':>9}")
for k in range(2, 7):
    bl = hierarchy.fcluster(_cz, k, criterion='maxclust')
    sil = silhouette_score(Z, bl)
    ari = adjusted_rand_score(_gj.astype(int), bl)
    null = np.array([adjusted_rand_score(_gj[_rj.permutation(len(_gj))].astype(int), bl)
                     for _ in range(1000)])
    print(f"    {k:>3}{sil:>13.3f}{ari:>16.3f}"
          f"{(np.sum(null >= ari) + 1) / 1001:>9.3f}")
print("    Silhouette always returns a best k, so a peak here is not evidence of")
print("    clusters; the ARI column is the one that speaks to whether behavioural")
print("    structure lines up with the neural one.")

# ---- figure ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6))
PAL_J = ['#1D3557', '#C1121F']


def _scatter(ax, xk, yk):
    x, y = FEATS[xk], FEATS[yk]
    for gi, (mask, nm) in enumerate(((~_gj, _keep_j[0]), (_gj, _keep_j[1]))):
        ax.scatter(x[mask], y[mask], s=42, color=PAL_J[gi], alpha=.75,
                   edgecolor='white', linewidth=.6,
                   label=f"cluster {nm} (n={mask.sum()})")
        # centroid, and the 1-sd ellipse of that cluster's own covariance
        mu = np.array([x[mask].mean(), y[mask].mean()])
        ax.plot(*mu, marker='X', ms=13, color=PAL_J[gi], mec='k', mew=1.1, zorder=6)
        w, V = np.linalg.eigh(np.cov(np.column_stack([x[mask], y[mask]]).T))
        th = np.linspace(0, 2 * np.pi, 100)
        E = V @ (np.sqrt(np.maximum(w, 0))[:, None]
                 * np.array([np.cos(th), np.sin(th)]))
        ax.plot(E[0] + mu[0], E[1] + mu[1], color=PAL_J[gi], lw=1.2, alpha=.7)
    _pj = _joint.get(f"{xk} + {yk}", np.nan)
    ax.set_xlabel(xk)
    ax.set_ylabel(yk)
    ax.set_title(f"{xk} x {yk}\njoint p = {_pj:.3f}   "
                 f"({xk} alone: p = {_uni[xk]:.3f})", fontsize=10)
    ax.legend(frameon=False, fontsize=9)
    sns.despine(ax=ax)


for _ax, _pair in zip(axes[:2], PAIRS):
    _scatter(_ax, *_pair)

ax = axes[2]
ks = list(FEATS)
dvals = [(FEATS[k][_gj].mean() - FEATS[k][~_gj].mean()) /
         np.sqrt((FEATS[k][_gj].var(ddof=1) + FEATS[k][~_gj].var(ddof=1)) / 2)
         for k in ks]
ax.barh(np.arange(len(ks)), dvals, color='0.55', edgecolor='white', height=.6)
for j, k in enumerate(ks):
    ax.text(dvals[j] + (.02 if dvals[j] >= 0 else -.02), j,
            f"p = {_uni[k]:.3f}", va='center',
            ha='left' if dvals[j] >= 0 else 'right', fontsize=8)
ax.axvline(0, color='k', lw=.8)
ax.set_yticks(np.arange(len(ks)))
ax.set_yticklabels(ks, fontsize=9)
ax.set_xlabel("Cohen's d  (cluster 2 - cluster 1)")
ax.set_xlim(min(dvals) - .40, max(dvals) + .40)
ax.set_title("one summary at a time", fontsize=10)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('figure_ibl/IBL_behaviour_space.svg', dpi=300, bbox_inches='tight')
plt.savefig('figure_ibl/IBL_behaviour_space.pdf', bbox_inches='tight')
plt.show()

In [ ]:
### COMBINED FIGURE - all panels above, assembled into one figure.
from pathlib import Path

from matplotlib.patches import Patch

# figures 1 and 2's plotting module, so panel h matches them
import shapemetrics as sm          # noqa: E402
from shapemetrics import plotting

# display names for the genotype codes (labels only, no analysis)
_gname = {'N': 'wild type', 'F': 'Fmr1', 'C': 'Cntnap2', 'S': 'Shank3'}

# The three mutant lines are drawn in one grey: the figure's claim is wild type
# against the rest (panel d decodes only wild type above chance), not one mutant
# against another, and three separate hues implied a distinction the analysis
# does not make.  `colors` itself is left alone -- other cells use it.
_MUT_GREY = "0.62"
colors = {**colors, 'F': _MUT_GREY, 'C': _MUT_GREY, 'S': _MUT_GREY}

# Uses only variables already computed by the cells above; no new analysis.
# Run the notebook top to bottom first, then this cell.

# one place to retune every font size in this figure
FS_TITLE, FS_LABEL, FS_TICK = 14, 13, 12
FS_SMALL, FS_LEG, FS_LETTER = 11, 11, 17
FS_STAR = 15     # panel a's cells are full-panel now, so '***' has room

CONTRASTS = np.array([-100., -25., -12.5, -6.25, 0., 6.25, 12.5, 25., 100.])
CL_PALETTE = ['#1D3557', '#C1121F', '#2A9D8F', '#E9A03B']
MIN_N_MEAN = 3          # clusters smaller than this are drawn mouse by mouse
K_CLUST = 3

fig = plt.figure(figsize=(16.5, 7.6))
gs = fig.add_gridspec(2, 5, hspace=.55, wspace=.42,
                      left=.07, right=.97, top=.94, bottom=.06)
TEAL, RED, GREY = "#2A9D8F", "#C1121F", "0.55"
BOX = dict(boxstyle="round,pad=0.28", fc="white", ec="none", alpha=.82)


def _cbar(ax, im, width=.045, pad=.04):
    '''Bare colour key beside `ax`: the ramp, no ticks, no frame, no label.

    NOTE the two matrices it is used on do not share a direction.  Panel b is
    drawn with cmap="gray" (black = SMALL distance) and panel e with cmap="Greys"
    (black = LARGE distance), so the two bars look alike and read opposite.
    '''
    cax = ax.inset_axes([1 + pad, 0, width, 1])
    cb = ax.figure.colorbar(im, cax=cax)
    cb.set_ticks([])
    cb.outline.set_visible(False)
    return cb


def _sq(ax):
    ax.set_box_aspect(1)
    return ax


def _stars(p):
    """conventional significance markers from a p-value"""
    if p is None or np.isnan(p):
        return ''
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ''


def _dark(col, f=0.55):
    """darker shade of a colour -- a marker filled with the violin's own colour
    is invisible against it"""
    r_, g_, b_ = mcolors.to_rgb(col)
    return (r_ * f, g_ * f, b_ * f)


def _tag(ax, letter, dx=-0.18, dy=1.06):
    ax.text(dx, dy, letter, transform=ax.transAxes, fontweight='bold', fontsize=FS_LETTER)


# ---- a  reserved -----------------------------------------------------------
# Left deliberately empty: the dataset schematic goes here. The axes exists only
# to hold the panel letter and to keep the grid spacing honest.
_a = fig.add_subplot(gs[0, 0])
_a.axis('off')
_tag(_a, 'a', dx=-0.06, dy=1.02)

# ---- e  distances across regions -------------------------------------------
# The visible block is the 4 x 4 upper triangle, so with an equal aspect it
# fills the square panel exactly -- no host axes or inset needed.
ax = _sq(fig.add_subplot(gs[1, 0]))
# cell 8 sets the lower triangle + diagonal of means_dist to NaN in place, so
# re-derive the full matrix with that cell's own first line
means_dist_full = np.nanmean(means_dist_subject, axis=0)
Mup = means_dist_full.copy()
Mup[np.tril_indices(len(regions))] = np.nan
# black-to-grey, saturating at d = 10
_im_reg = ax.imshow(Mup, cmap='Greys', vmin=0, vmax=10)
for r in range(len(regions)):
    for c in range(len(regions)):
        if r < c:
            ax.text(c, r, _stars(p_cell[r, c]), ha='center', va='center',
                    fontsize=FS_STAR, color='red')
ax.set_xticks(range(len(regions)))
ax.set_xticklabels(regions, fontsize=FS_TICK, rotation=0, ha='center')
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
ax.set_yticks(range(len(regions)))
ax.set_yticklabels(regions, fontsize=FS_TICK)
ax.yaxis.set_ticks_position('right')
ax.yaxis.set_label_position('right')
ax.set_xlim(0.5, len(regions) - 0.5)
ax.set_ylim(len(regions) - 1.5, -0.5)
# Horizontal key under the matrix.  cmap="Greys" here, so BLACK is a LARGE
# distance: the ends are labelled rather than ticked, since the values saturate
# at vmax = 10 and the absolute numbers carry little.
_cax = ax.inset_axes([0.0, -0.16, 1.0, 0.055])
_cb = fig.colorbar(_im_reg, cax=_cax, orientation="horizontal")
_cb.set_ticks([]); _cb.outline.set_visible(False)
_cax.text(-0.02, 0.5, "low", transform=_cax.transAxes, ha="right", va="center",
          fontsize=FS_SMALL, color="0.25")
_cax.text(1.02, 0.5, "high", transform=_cax.transAxes, ha="left", va="center",
          fontsize=FS_SMALL, color="0.25")
_cax.text(0.5, -1.0, "distance", transform=_cax.transAxes, ha="center",
          va="top", fontsize=FS_SMALL, color="0.25")
ax.set_title("distances across regions", fontsize=FS_TITLE, pad=22)
for _sp in ax.spines.values():
    _sp.set_visible(False)
ax.tick_params(length=0)
_tag(ax, 'e', dy=1.16)

# ---- f  psychometric curves ----------------------------------------------
ax = _sq(fig.add_subplot(gs[1, 1]))
ax.plot(choices.T, color=GREY, alpha=.35, lw=.7)
ax.plot(np.nanmean(choices, 0), color=TEAL, lw=2.5)
ax.set_xticks(range(9))
ax.set_xticklabels([f"{v:g}" for v in CONTRASTS],
                   rotation=45, fontsize=FS_TICK)
ax.set_yticks([0, .5, 1]); ax.set_ylim(0, 1)
ax.set_xlabel("stimulus contrast (%)"); ax.set_ylabel("P(right)")
ax.set_title(f"behaviour, {choices.shape[0]} mice", fontsize=FS_TITLE)
_tag(ax, 'f')
sns.despine(bottom=False, left=False, ax=ax)

# ---- g  distance matrices -------------------------------------------------
ax = _sq(fig.add_subplot(gs[1, 2]))
ax.imshow(dist_neural_, cmap="Grays")
ax.imshow(dist_cc_, cmap="Purples")
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("pairwise distances", fontsize=FS_TITLE)
# which triangle is which, said on the axes rather than in a caption below
ax.set_ylabel("neural", fontsize=FS_TITLE, color="0.3")
_axr = ax.twinx()
_axr.set_ylim(ax.get_ylim())
_axr.set_yticks([])
# a twin does not inherit box_aspect, and it shares (and so overrides) the
# position of the axes it twins -- without this the panel stops being square
_axr.set_box_aspect(1)
_axr.set_ylabel("behaviour", fontsize=FS_TITLE, color="#5B3A8E")
for _sp in _axr.spines.values():
    _sp.set_visible(False)
_tag(ax, 'g', dx=-0.06)

# ---- h  scatter of distances ----------------------------------------------
ax = _sq(fig.add_subplot(gs[1, 3]))
_x = np.asarray(flat_dist_cc, dtype=float)
_y = np.asarray(flat_dist_neural, dtype=float)
ax.scatter(_x, _y, color="k", facecolors='gray', s=18, alpha=0.35, linewidths=0.4)
_b1, _b0 = np.polyfit(_x, _y, 1)
_xs = np.linspace(_x.min(), _x.max(), 20)
ax.plot(_xs, _b1 * _xs + _b0, color='red', lw=1.5, zorder=8)
ax.text(0.04, 0.96, "r={:.2f}\np={:.2g}".format(r_dist, p_dist), color="red",
        transform=ax.transAxes, va='top', fontsize=FS_LABEL, bbox=BOX)
ax.set_xlabel("behavioral\ndistance", fontsize=FS_TITLE)
ax.set_ylabel("neural\ndistance", fontsize=FS_TITLE)
ax.set_xticks([]); ax.set_yticks([])
_tag(ax, 'h')
sns.despine(bottom=False, left=False, ax=ax)

# ---- i  per-region z-scores, plus the pooled distribution -----------------
ax = _sq(fig.add_subplot(gs[1, 4]))
z_df = pd.concat([region_info[["region", "z-score"]],
                  pd.DataFrame({"region": "all", "z-score": np.asarray(z).ravel()})],
                 ignore_index=True)
order_e = list(regions) + ["all"]
pal_e = dict(palette); pal_e["all"] = "#4C4C4C"
sns.violinplot(x="region", y="z-score", data=z_df, order=order_e, palette=pal_e,
               ax=ax, inner=None, linewidth=1)
ax.axhline(0, color='k', linewidth=.8)
for i, name in enumerate(order_e):
    v = z_df.loc[z_df.region == name, "z-score"].values
    lo, hi = np.nanpercentile(v, [2.5, 97.5])
    ax.plot([i, i], [lo, hi], color='k', lw=1.1, zorder=5, solid_capstyle='butt')
    ax.plot([i - .07, i + .07], [lo, lo], color='k', lw=1.1, zorder=5)
    ax.plot([i - .07, i + .07], [hi, hi], color='k', lw=1.1, zorder=5)
    ax.plot(i, np.nanmean(v), 'o', mfc=_dark(pal_e[name]), mec='k',
            mew=0.8, ms=7, zorder=6)
ax.set_ylabel("z-score", fontsize=FS_TITLE)
ax.set_xlabel("region", fontsize=FS_TITLE)
ax.tick_params(labelsize=FS_TICK)
ax.set_title("behaviour-geometry correlation\n(shuffle corrected)",
             fontsize=FS_TITLE)
_tag(ax, 'i')
sns.despine(bottom=False, left=False, ax=ax)

# ---- b  genotype: ordered distance matrix + dendrogram --------------------
sub = gs[0, 1].subgridspec(2, 1, height_ratios=[1, 4], hspace=.05)
axd = fig.add_subplot(sub[0])
hierarchy.dendrogram(linkage, ax=axd, no_labels=True, color_threshold=0,
                     above_threshold_color="k")
axd.axis("off")
axd.set_title("genotype clustering", fontsize=FS_TITLE)
_tag(axd, 'b', dx=-0.10, dy=1.25)
axh = fig.add_subplot(sub[1])
axh.imshow(ordered_dist, cmap="gray", aspect="auto")
axh.set_xticks([]); axh.set_yticks([])
strip = mcolors.to_rgba_array([colors[lab] for lab in labels_arr[leaf_order]])
axh.imshow(strip.reshape(1, -1, 4), aspect="auto", origin="lower",
           extent=(-0.5, len(leaf_order) - 0.5, -2.8, -0.6))
axh.set_ylim(-2.8, len(leaf_order) - 0.5)
# a bare colour key: the distances have no interpretable absolute scale here,
# so the bar shows the ramp and nothing else
axh.legend(handles=[Patch(facecolor=colors['N'], label='wild type'),
                    Patch(facecolor=_MUT_GREY, label='mutants')],
           frameon=False, fontsize=FS_LEG, ncol=2, loc="upper center",
           bbox_to_anchor=(.5, -.04), handlelength=1, columnspacing=0.7)

# ---- c  do the animals fall into groups, and is it the genotypes? -----------
# Two runs of the same test under one letter, differing only in which animals
# are in it.  Left: all 36.  Right: the 26 mutants alone.  If the lumpiness of
# the full set were the genotypes separating, it should survive in a set that
# still contains three of them; it does not, which is what makes the pair worth
# showing together rather than either alone.
#
# Drawn by `Posani/code/plotting.py`, the module figures 1 and 2 use, so the
# palette matches them (blue is always the null, firebrick always the data) with
# the type sizes set to this figure's larger scale.  Numbers come from the cache
# written by asd_subject_gaussian.ipynb; this cell draws and does not recompute.
for _j, (_id, _ttl, _lab) in enumerate((("all", "all genotypes", "animals"),
                                        ("mutants", "mutants", "mutants"))):
    _axg = fig.add_subplot(gs[0, 2 + _j])
    _z = np.load(paths.results("results_asd") / f"subject_gaussian_{_id}.npz")
    _r = sm.null_stats(_z["obs_sweep"].max(), _z["null_sweep"].max(1))
    plotting.null_hist(_axg, _r["obs"], _r["null"], "best silhouette",
                       label_null="one blob", label_obs=_lab, p=_r["p"],
                       strip_xticks=True)
    # two lines: at this panel width a single-line title runs into its
    # neighbour and into the panel letter
    _axg.set_title(f"{_ttl}\n(n = {len(_z['emb'])})", fontsize=FS_TITLE)
    _axg.xaxis.label.set_fontsize(FS_LABEL)
    _lgg = _axg.get_legend()
    if _lgg is not None:
        for _t in _lgg.get_texts():
            _t.set_fontsize(FS_LEG)
    if _j == 0:
        _tag(_axg, 'c', dx=-0.22, dy=1.28)
    print(f"panel h / {_ttl:<13} n = {len(_z['emb']):>2}  silhouette {_r['obs']:.4f}"
          f" vs {_r['null'].mean():.4f} +/- {_r['null'].std():.4f}"
          f"   z = {_r['z']:+.2f}, p = {_r['p']:.4f}"
          f"   (floor {1 / (len(_r['null']) + 1):.4f})")

# ---- d  genotype decoding ---------------------------------------------------
# Bars come straight from `res`, computed in the decoding cell above -- this
# cell only draws them, so re-running the figure costs nothing. Each contrast is
# one genotype against all remaining animals, decoded leave-one-out by nearest
# group in shape space and scored by balanced accuracy, so chance is 0.5
# regardless of how unbalanced the two sides are.
ax = _sq(fig.add_subplot(gs[0, 4]))
_acc = [r['acc'] for r in res]
_xg = np.arange(len(res))
ax.bar(_xg, _acc, width=.62, color=[colors[r['key']] for r in res],
       edgecolor='white', linewidth=.6, zorder=3)
ax.errorbar(_xg, _acc, yerr=[r['sem'] for r in res], fmt='none', ecolor=RED,
            elinewidth=1.2, capsize=3.5, capthick=1.2, zorder=5)
ax.axhline(.5, color='k', ls='--', lw=1, zorder=2)
_top = max(a + r['sem'] for a, r in zip(_acc, res))
for j, r in enumerate(res):
    # stars where the permutation test clears the usual thresholds; the exact
    # p values are printed below and belong in the caption
    ax.text(j, r['acc'] + r['sem'] + .015 * _top, _stars(r['p']), ha='center',
            va='bottom', fontsize=FS_LABEL, color='k')
# between the two shortest bars is the only clear space above the line
ax.text(1.5, .505, "chance", ha='center', va='bottom', fontsize=FS_SMALL,
        color='0.35')
ax.set_ylim(.3, _top + .12)
ax.set_xticks(_xg, [_gname[r['key']] for r in res], rotation=30,
              fontsize=FS_LABEL)
ax.set_ylabel("balanced accuracy", fontsize=FS_TITLE)
ax.set_title("genotype decoding", fontsize=FS_TITLE)
_tag(ax, 'd')
sns.despine(bottom=False, left=False, ax=ax)

fig.savefig("figure_ibl/IBL_combined_all.pdf", bbox_inches="tight")
fig.savefig("figure_ibl/IBL_combined_all.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved figure_ibl/IBL_combined_all.pdf")
print("panel i, leave-one-out decoding vs %d label shuffles:" % N_PERM)
for r in res:
    print(f"  {r['name']:<24} balanced acc {r['acc']:.3f} "
          f"+/- {r['sem']:.3f}   p = {r['p']:.4f}")